# Hindi 25M LM - evaluation, reasoning finetuning, attention analysis
Settings: **GPU T4**, **Internet On**. Inputs to add: the `hindi-tokens` dataset and the **output of the pretraining
notebook** (Add Input -> Notebook Output) - the checkpoint is found automatically.
Everything is written to `/kaggle/working/report/` and `/kaggle/working/runs/`.

In [ ]:
import subprocess, os, glob, shutil
if not os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git','clone','https://github.com/Keerthi-2012/Monolingual-Transformer-LM.git','/kaggle/working/repo'],check=True)
else:
    subprocess.run(['git','-C','/kaggle/working/repo','pull'],check=True)
print(subprocess.run(['git','-C','/kaggle/working/repo','log','-1','--oneline'],capture_output=True,text=True).stdout)
!pip -q install sacrebleu
!apt-get -qq install -y fonts-noto-core > /dev/null 2>&1; fc-cache -f > /dev/null 2>&1
TOKENS = os.path.dirname(glob.glob('/kaggle/input/**/train_000.bin', recursive=True)[0])
CK = sorted(glob.glob('/kaggle/input/**/ckpt_best.pt', recursive=True), key=os.path.getmtime)[-1]
print('tokens:', TOKENS); print('checkpoint:', CK)
R = '/kaggle/working/report'; os.makedirs(f'{R}/attn', exist_ok=True)
src = os.path.dirname(CK)
for f in ('train_log.jsonl','loss_curve.png','config.json'):      # keep the pretraining logs with the report
    if os.path.exists(f'{src}/{f}'): shutil.copy(f'{src}/{f}', R)

## 1. Pretrained model: perplexity / bits-per-byte, generation metrics, attention

In [ ]:
!cd /kaggle/working/repo/hindi/eval && python lm_eval.py --ckpt {CK} --tokens {TOKENS} --out {R}/lm_eval.json

In [ ]:
!cd /kaggle/working/repo/hindi/eval && python gen_eval.py --ckpt {CK} --tokens {TOKENS} --out-prefix {R}/gen

In [ ]:
!cd /kaggle/working/repo/hindi/eval && python attn_analysis.py --ckpt {CK} --tokens {TOKENS} --out-dir {R}/attn --name pretrained

## 2. Reasoning finetuning (zero-shot baseline -> finetune -> evaluate)

In [ ]:
# the reasoning dataset is deterministic and not stored in git: generate it here (a few seconds)
!cd /kaggle/working/repo/hindi/reasoning && python generate.py --out data

In [ ]:
!cd /kaggle/working/repo/hindi/reasoning && python -u finetune.py --pretrained {CK} --out-dir /kaggle/working/runs/ft

In [ ]:
FT = '/kaggle/working/runs/ft'
for f in glob.glob(f'{FT}/acc_*.json') + [f'{FT}/finetune_log.jsonl']:
    shutil.copy(f, R)
os.makedirs(f'{R}/preds', exist_ok=True)
for f in glob.glob(f'{FT}/preds_*.jsonl'): shutil.copy(f, f'{R}/preds')

## 3. Attention analysis after finetuning (same toolkit, finetuned checkpoint)

In [ ]:
!cd /kaggle/working/repo/hindi/eval && python attn_analysis.py --ckpt {FT}/ckpt_finetuned.pt --tokens {TOKENS} --out-dir {R}/attn --name finetuned

In [ ]:
import json
for f in ('lm_eval.json','gen_metrics.json','acc_pretrained.json','acc_finetuned.json'):
    print('=====', f); print(json.dumps(json.load(open(f'{R}/{f}')), indent=1)[:3000])